# Machine Translation — the evaluation core, from scratch

This notebook is the runnable companion to the **[12 Machine Translation](../12-Machine-Translation.md)** concept page. It imports every function from the single source-of-truth module **`machine_translation.py`**, so the notebook, the page, and the figures all share one verified computation and cannot drift apart.

We focus on what is *specific to translation* and *derivable by hand*:

1. **BLEU from scratch** — modified n-gram precision, the brevity penalty, the geometric mean — **verified to match sacreBLEU and nltk** to floating-point precision.
2. **BLEU's brittleness** — a meaning-preserving paraphrase scores ~0 BLEU, motivating **chrF** (also from scratch, also matched to sacreBLEU).
3. **IBM Model 1** — word alignment learned by EM, with no dictionary.
4. **Beam-search length normalization** — why a longer faithful output beats a short truncation.
5. **A real NMT model** (`opus-mt-fr-en`) translated and scored.

Every cell **asserts its qualitative point before printing**, so a silent regression fails loudly. Run top to bottom.

## Setup — import the source-of-truth module and pin the environment

The core math is pure-numpy. We print an honest device/version line and a **stable md5 hash** of the module text (never Python's salted `hash()`), so two runs of the same code produce the same fingerprint.

In [1]:
import hashlib
import sys
from pathlib import Path

# Import the canonical functions — the notebook computes NOTHING that the .py doesn't define.
import machine_translation as mt

print(mt.environment_line())

# Stable content fingerprint of the module (md5 of the file text), NOT Python's salted hash().
module_path = Path(mt.__file__)
module_md5 = hashlib.md5(module_path.read_text(encoding="utf-8").encode("utf-8")).hexdigest()
print(f"machine_translation.py md5: {module_md5}")
print(f"python: {sys.version.split()[0]}")

Python 3.12.13 | numpy 2.4.6 | torch 2.12.0 | device: cpu (the from-scratch math is integer/float counting — CPU is exact)
machine_translation.py md5: 9bc975646d3bb8772cd38be15bc00794
python: 3.12.13


## 1. BLEU from scratch — and the proof it matches sacreBLEU and nltk

BLEU = $\mathrm{BP}\cdot\exp(\sum_n w_n \log p_n)$, built from:
- **modified (clipped) n-gram precision** $p_n$ — a candidate n-gram is credited at most as often as it appears in any single reference, so repetition can't cheat;
- the **brevity penalty** $\mathrm{BP}$ — punishes output shorter than the reference (BLEU's stand-in for recall);
- the **geometric mean** of $p_1..p_4$ — zero if *any* $p_n$ is zero.

We compute it on a worked example and print every intermediate term.

In [2]:
cand = "the the the black cat sat on the mat happily today"
ref = "the black cat sat on the mat very happily today indeed"

res = mt.sentence_bleu(cand, [ref])

# Assert the clipping actually fired: the candidate has 3x "the" but the reference has 1,
# so the clipped 1-gram numerator must be below the raw candidate "the" count.
assert res["clipped"][0] == 9 and res["total"][0] == 11, "p_1 clipping changed unexpectedly"
assert res["bp"] == 1.0, "candidate is as long as the reference, so BP must be 1"

for n in range(mt.MAX_N):
    print(f"  p_{n+1} = {res['clipped'][n]:>2}/{res['total'][n]:>2} = {res['precisions'][n]:5.1f}%")
print(f"  brevity penalty BP = {res['bp']:.4f}  (cand_len={res['cand_len']}, ref_len={res['ref_len']})")
print(f"  geometric mean of precisions = {res['geo_mean']:.4f}")
print(f"  BLEU = BP * geo_mean * 100 = {res['score']:.6f}")

  p_1 =  9/11 =  81.8%
  p_2 =  7/10 =  70.0%
  p_3 =  5/ 9 =  55.6%
  p_4 =  4/ 8 =  50.0%
  brevity penalty BP = 1.0000  (cand_len=11, ref_len=11)
  geometric mean of precisions = 0.6316
  BLEU = BP * geo_mean * 100 = 63.155524


### Verify against the reference implementations

The from-scratch BLEU is **not** an approximation. On whitespace-tokenized input it reproduces **sacreBLEU** (`tokenize='none'`) and **nltk** `corpus_bleu` to floating-point precision. We assert the match.

In [3]:
from sacrebleu.metrics import BLEU
from nltk.translate.bleu_score import corpus_bleu as nltk_corpus_bleu

cands = [cand, "machine translation has progressed a lot"]
refs = [[ref], ["machine translation has progressed a lot"]]

mine = mt.corpus_bleu(cands, refs)["score"]
sacre = BLEU(tokenize="none", effective_order=False).corpus_score(cands, list(zip(*refs))).score
nltk_score = nltk_corpus_bleu([[r[0].split()] for r in refs], [c.split() for c in cands]) * 100

# The three implementations must agree to floating-point tolerance.
assert abs(mine - sacre) < 1e-9, f"scratch vs sacreBLEU diverged: {mine} vs {sacre}"
assert abs(mine - nltk_score) < 1e-9, f"scratch vs nltk diverged: {mine} vs {nltk_score}"

print(f"scratch BLEU : {mine:.10f}")
print(f"sacreBLEU    : {sacre:.10f}")
print(f"nltk BLEU    : {nltk_score:.10f}")
print("all three agree to < 1e-9  ->  the derivation is trustworthy")

scratch BLEU : 74.6765437238
sacreBLEU    : 74.6765437238
nltk BLEU    : 74.6765437238
all three agree to < 1e-9  ->  the derivation is trustworthy


## 2. BLEU's brittleness — a correct paraphrase scores ~0

This is the single most important fact about MT evaluation. A *perfect* translation worded differently from the reference shares almost no word n-grams, so **BLEU collapses to 0**. **chrF** — the same idea over *character* n-grams — still credits the overlap, because words like "tuesday" and "the" share many characters even when whole words differ. (Our `chrF` matches sacreBLEU exactly.)

In [4]:
import sacrebleu

reference = "the committee will convene on tuesday to discuss the budget"
exact = "the committee will convene on tuesday to discuss the budget"
paraphrase = "the panel meets tuesday to talk about the finances"

bleu_exact = mt.sentence_bleu(exact, [reference])["score"]
bleu_para = mt.sentence_bleu(paraphrase, [reference])["score"]
chrf_exact = mt.chrf(exact, reference)
chrf_para = mt.chrf(paraphrase, reference)

# The qualitative claims, asserted before we print them:
assert bleu_exact == 100.0, "exact match must score BLEU 100"
assert bleu_para < 1.0, "the paraphrase shares no 4-grams, so BLEU must be ~0"
assert chrf_para > bleu_para, "chrF must credit the paraphrase more than BLEU does"
# And chrF must match sacreBLEU's chrF exactly.
assert abs(chrf_para - sacrebleu.sentence_chrf(paraphrase, [reference]).score) < 1e-6

print(f"  exact match      : BLEU={bleu_exact:6.2f}   chrF={chrf_exact:6.2f}")
print(f"  valid paraphrase : BLEU={bleu_para:6.2f}   chrF={chrf_para:6.2f}")
print("  a PERFECT translation scores 0 BLEU but partial chrF -- why the field moved past word overlap")

  exact match      : BLEU=100.00   chrF=100.00
  valid paraphrase : BLEU=  0.00   chrF= 26.88
  a PERFECT translation scores 0 BLEU but partial chrF -- why the field moved past word overlap


> **Predict, then run:** lower the BLEU n-gram order from 4 to 2 below. The paraphrase's BLEU climbs off the floor — fewer, shorter n-grams are easier to match. This is also why short-text BLEU needs smoothing.

In [5]:
bleu_para_n2 = mt.sentence_bleu(paraphrase, [reference], max_n=2)["score"]
assert bleu_para_n2 > bleu_para, "with only 1- and 2-grams, the paraphrase should score higher than at n=4"
print(f"paraphrase BLEU at n=4: {bleu_para:.2f}")
print(f"paraphrase BLEU at n=2: {bleu_para_n2:.2f}   <- off the floor, because some bigrams now match")

paraphrase BLEU at n=4: 0.00
paraphrase BLEU at n=2: 43.44   <- off the floor, because some bigrams now match


## 3. IBM Model 1 — word alignment learned by EM (no dictionary)

Statistical MT's beating heart. From a tiny parallel corpus and a **uniform** start, EM iterates soft co-occurrence counts (E-step) and renormalizes them (M-step). Because `la` co-occurs with `the` in *both* sentences while `maison` co-occurs with `house` in only one, `the` gets "explained away" by `la`, and the table sharpens to the correct word map — with **no dictionary and no labels**.

In [6]:
t = mt.ibm_model1()  # uses the default toy corpus

best = {f: max(t[f], key=t[f].get) for f in t}
# The learned alignment must be the linguistically correct one:
assert best["la"] == "the", "la should align to the"
assert best["maison"] == "house", "maison should align to house"
assert best["fleur"] == "flower", "fleur should align to flower"

for f in sorted(t):
    print(f"  {f:8s} -> {best[f]:8s}  (p={t[f][best[f]]:.2f})")

  fleur    -> flower    (p=0.96)
  la       -> the       (p=0.99)
  maison   -> house     (p=0.96)


### The alignment matrix

The posterior alignment for `la maison fleur` / `the house flower` — the same data the page's heatmap is built from. The bright diagonal is the discovered word map; on a real language pair it would *bend* wherever the languages reorder (which is what attention later learned).

In [7]:
matrix = mt.alignment_matrix("la maison fleur", "the house flower", t)

# The diagonal must dominate: each English word's strongest source link is the aligned French word.
import numpy as np
assert int(matrix[0].argmax()) == 0, "the <- la"
assert int(matrix[1].argmax()) == 1, "house <- maison"
assert int(matrix[2].argmax()) == 2, "flower <- fleur"

fr_words = "la maison fleur".split()
en_words = "the house flower".split()
print("          " + "  ".join(f"{w:>6s}" for w in fr_words))
for i, e in enumerate(en_words):
    print(f"  {e:8s} " + "  ".join(f"{v:6.2f}" for v in matrix[i]))

              la  maison   fleur
  the        0.92    0.00    0.00
  house      0.04    1.00    0.00
  flower     0.04    0.00    1.00


## 4. Beam-search length normalization — recovering the full translation

A decoder scores a candidate by its total log-probability, the sum of per-token log-probs. Every term is negative, so a **longer** sequence accumulates more negative log-prob *just by being longer* — raw beam search is biased toward short, truncated output. **Length normalization** divides by $L^\alpha$ (GNMT uses $\alpha\approx0.6$) to undo that bias.

Candidate A is a 4-token truncation; B is the 7-token full translation whose extra tokens are confident. Watch the winner flip as $\alpha$ grows.

In [8]:
rows = mt.beam_length_norm_demo()

# The pedagogical flip: A (short) wins at alpha=0, B (full) wins at alpha=0.6.
assert rows[0]["winner"].startswith("A"), "raw log-prob (alpha=0) should prefer the short truncation"
assert rows[1]["winner"].startswith("B"), "length normalization (alpha=0.6) should recover the full output"

print(f"  {'alpha':>6} | {'A (4 tok)':>12} | {'B (7 tok)':>12} | winner")
print("  " + "-" * 56)
for r in rows:
    print(f"  {r['alpha']:>6.1f} | {r['score_a']:>12.3f} | {r['score_b']:>12.3f} | {r['winner']}")

   alpha |    A (4 tok) |    B (7 tok) | winner
  --------------------------------------------------------
     0.0 |       -1.500 |       -1.770 | A (short, truncated)
     0.6 |       -0.653 |       -0.551 | B (full, faithful)
     1.0 |       -0.375 |       -0.253 | B (full, faithful)


## 5. A real neural MT model, translated and scored

Finally, a real Transformer NMT model (`Helsinki-NLP/opus-mt-fr-en`) translating three French sentences with beam search, scored by sacreBLEU. This needs `transformers` + `sacrebleu`; if they're missing the cell prints a note and skips. The numbers are exact and reproducible (beam decode is deterministic).

In [9]:
nmt = mt.run_real_nmt(beam_size=5)
if nmt is not None:
    # Sentence 3 is an exact paraphrase of its reference -> chrF must be 100.
    assert abs(nmt["chrf"][2] - 100.0) < 1e-6, "sentence 3 should be an exact match (chrF 100)"
    # The corpus scores should match the page's measured numbers (rounded to 1 dp).
    assert round(nmt["corpus_bleu"], 1) == 59.0, "corpus BLEU drifted from the page"
    assert round(nmt["corpus_chrf"], 1) == 79.0, "corpus chrF drifted from the page"
    for src, hyp, ref_s, cf in zip(mt.NMT_SRCS, nmt["hyps"], mt.NMT_REFS, nmt["chrf"]):
        print(f"  FR: {src}")
        print(f"   -> {hyp}   (ref: {ref_s})   chrF={cf:.1f}")
    print(f"\n  corpus BLEU = {nmt['corpus_bleu']:.1f}   corpus chrF = {nmt['corpus_chrf']:.1f}")
else:
    print("real-NMT block skipped (install transformers sentencepiece sacremoses sacrebleu to run it)")

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

  FR: Le chat noir dort sur le canapé.
   -> The black cat sleeps on the couch.   (ref: The black cat is sleeping on the couch.)   chrF=67.7
  FR: J'aime apprendre les langues étrangères.
   -> I like to learn foreign languages.   (ref: I love learning foreign languages.)   chrF=65.1
  FR: La traduction automatique a beaucoup progressé.
   -> Machine translation has progressed a lot.   (ref: Machine translation has progressed a lot.)   chrF=100.0

  corpus BLEU = 59.0   corpus chrF = 79.0


## Recap

- **BLEU**, derived from scratch, reproduces sacreBLEU and nltk to floating-point precision — so you can trust the derivation, not just the number.
- A **valid paraphrase scores 0 BLEU** but partial **chrF**: the core reason MT evaluation climbed from word-overlap metrics to character-level and then learned (COMET/BLEURT) metrics.
- **IBM Model 1** discovers word alignments from co-occurrence alone — the soft-alignment ancestor of attention.
- **Length normalization** turns a short truncation into the full faithful translation — the MT decoding gotcha.
- A real **opus-mt-fr-en** model translates and scores exactly as the page reports.

Every claim above was **asserted before it was printed**. See the [concept page](../12-Machine-Translation.md) for the full narrative and the [references](../12-Machine-Translation.references.md) for sources.